# fMRIPrep → BrainVoyager conversion (high-level task)

Adapts the SFlow conversion to the high-level task with 3 runs (Faces / Bodies / Houses) randomized across subjects. Each run has 3 SF conditions: LSF, HSF, NSF.

Produces per subject × run:
- `*_confounds.sdm`  — nuisance regressors from fMRIPrep confounds TSV
- `*.prt`            — protocol with 3 condition predictors named `<SF>-<Category>`
                       (e.g., `LSF-Body`, `HSF-Body`, `NSF-Body`)

Across all 3 runs per subject, this produces 9 unique condition names: `{LSF,HSF,NSF} × {Body,House,Face}`. These are consistent across subjects regardless of the randomized run order, which is what BV's multi-study GLM needs.

**VTC files** are not generated here — import them manually via BV's File → Open NIfTI workflow as we did for the SFlow task. The bvbabel VTC writer had stripe artifacts that we never resolved.

## 1. Install / import

In [2]:
from pathlib import Path
import re
import bvbabel
import numpy as np
import pandas as pd

## 2. Configuration

In [3]:
# --- Paths ---------------------------------------------------------------
FMRIPREP_DIR = Path("/Volumes/drive/AVP-BDD/derivatives")
BEHAV_DIR    = Path("/Volumes/drive/AVP-BDD/behavior")
BV_DIR       = Path("/Volumes/drive/AVP-BDD/derivatives/brainvoyager")
BV_DIR.mkdir(parents=True, exist_ok=True)

# --- Subjects & runs -----------------------------------------------------
SUBJECTS = [
    102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
    112, 113, 115, 116, 117, 118, 119, 120, 121, 122,
    123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
    201, 202, 204, 205, 206, 207, 208, 209, 210, 211,
    212, 213, 214, 215, 216, 217, 218, 219, 220, 221,
    222, 223, 224, 225, 226, 227, 228, 229, 230,
]
RUNS = [1, 2, 3]

# --- Acquisition / task labels -------------------------------------------
TR_SECONDS = 1.0
TASK_LABEL = "SFhigh"             # <-- change if the high-level task uses a different BIDS label
SPACE      = "MNI152NLin2009cAsym"

# --- High-level task structure -------------------------------------------
# Categories appear in behavioural filenames: Subject_<id>_Run_<r>_<Category>_RT.csv
CATEGORIES = ["Body", "Face", "House"]      # high-level stimulus categories
SF_LEVELS  = ["LSF", "HSF", "NSF"]           # SF filtering levels in the CSV

# --- Confounds to include in SDM -----------------------------------------
MOTION_COLS         = ["trans_x", "trans_y", "trans_z",
                       "rot_x",   "rot_y",   "rot_z"]
EXTRA_COLS          = ["framewise_displacement"]
ACOMPCOR_N          = 6
COSINE_PREFIX       = "cosine"
ADD_MOTION_OUTLIERS = True

# --- Condition colours for the 9 SF × Category combinations -------------
# (R, G, B) in 0-255. Tweak however you like; just used for BV's predictor display.
CATEGORY_BASE_RGB = {
    "Body":  (200,  80,  80),    # red-ish
    "House": (80,  200,  80),    # green-ish
    "Face":  (80,   80, 200),    # blue-ish
}
SF_BRIGHTNESS = {
    "LSF": 0.6,
    "NSF": 0.85,
    "HSF": 1.1,
}
def predictor_color(sf_level, category):
    base = CATEGORY_BASE_RGB[category]
    f    = SF_BRIGHTNESS[sf_level]
    return tuple(int(min(255, c * f)) for c in base)

print(f"Will process {len(SUBJECTS)} subjects × {len(RUNS)} runs = "
      f"{len(SUBJECTS) * len(RUNS)} run(s)")
print(f"Task: task-{TASK_LABEL}")
print(f"Output: {BV_DIR}")

Will process 59 subjects × 3 runs = 177 run(s)
Task: task-SFhigh
Output: /Volumes/drive/AVP-BDD/derivatives/brainvoyager


## 3. PRT conversion (with category extraction)

Auto-discovers the category from the behavioural CSV filename (`Subject_<id>_Run_<r>_<Category>_RT.csv`). Names predictors `<SF>-<Category>` so the 9 unique conditions match across subjects regardless of run-order randomization.

In [4]:
BEHAV_PATTERN = re.compile(
    r"Subject_(?P<sub>\d+)_Run_(?P<run>\d+)_(?P<category>[A-Za-z]+)_RT\.csv$"
)


def find_behav_csv(behav_dir: Path, sub_num: int, run: int):
    """Find the high-level RT CSV for this sub/run; return (path, category).

    The filename embeds the category for that run, which differs across
    subjects (run order randomized)."""
    raw_id = str(sub_num)
    sub_dir = behav_dir / raw_id / "high-level"
    if not sub_dir.exists():
        return None, None
    for f in sub_dir.iterdir():
        m = BEHAV_PATTERN.match(f.name)
        if not m:
            continue
        if int(m.group("sub")) == sub_num and int(m.group("run")) == run:
            return f, m.group("category")
    return None, None


def behav_to_prt(csv_path: Path, prt_path: Path, category: str) -> int:
    """Convert a high-level RT CSV into a BV PRT.
    Each block becomes one epoch named '<SF>-<Category>'. Returns # blocks written."""
    df = pd.read_csv(csv_path)

    blocks = (df.groupby("Block")
                .agg(onset=("Stimulus Onset (s)",  "min"),
                     offset=("Stimulus Offset (s)", "max"),
                     condition=("Condition",        "first"),
                     n_conds=("Condition", lambda x: x.nunique()))
                .reset_index())

    if (blocks["n_conds"] != 1).any():
        bad = blocks[blocks["n_conds"] != 1]["Block"].tolist()
        raise ValueError(f"Block(s) {bad} contain multiple SF levels in {csv_path}")

    unknown = sorted(set(blocks["condition"]) - set(SF_LEVELS))
    if unknown:
        raise ValueError(f"Unexpected SF level(s) {unknown} in {csv_path}")

    blocks["label"]     = blocks["condition"] + "-" + category
    blocks["onset_ms"]  = (blocks["onset"]  * 1000.0).round().astype(int)
    blocks["offset_ms"] = (blocks["offset"] * 1000.0).round().astype(int)

    cond_list = []
    for sf in SF_LEVELS:
        label = f"{sf}-{category}"
        sub = blocks[blocks["label"] == label].sort_values("onset_ms")
        if len(sub) == 0:
            continue
        intervals = sub[["onset_ms", "offset_ms"]].to_numpy(dtype=np.int32)
        cond_list.append({
            "NameOfCondition": label,
            "NrOfOccurances":  len(sub),
            "Time start":      intervals[:, 0].tolist(),
            "Time stop":       intervals[:, 1].tolist(),
            "Color":           list(predictor_color(sf, category)),
        })

    header = {
        "FileVersion":         2,
        "ResolutionOfTime":    "msec",
        "Experiment":          prt_path.stem,
        "BackgroundColor":     "0 0 0",
        "TextColor":           "255 255 255",
        "TimeCourseColor":     "255 255 255",
        "TimeCourseThick":     3,
        "ReferenceFuncColor":  "0 0 80",
        "ReferenceFuncThick":  3,
        "NrOfConditions":      len(cond_list),
    }
    bvbabel.prt.write_prt(str(prt_path), header, cond_list)
    return len(blocks)

## 4. SDM conversion (confound regressors)

Identical to the SFlow notebook — just operates on the confounds TSV from the high-level task.

In [5]:
def confounds_to_sdm(tsv_path: Path, sdm_path: Path) -> int:
    """Build an SDM of nuisance regressors from fMRIPrep confounds TSV."""
    df = pd.read_csv(tsv_path, sep="\t")

    cols = list(MOTION_COLS) + list(EXTRA_COLS)
    cols += sorted(c for c in df.columns if c.startswith("a_comp_cor_"))[:ACOMPCOR_N]
    cols += sorted(c for c in df.columns if c.startswith(COSINE_PREFIX))
    if ADD_MOTION_OUTLIERS:
        cols += sorted(c for c in df.columns if c.startswith("motion_outlier"))
    cols = [c for c in cols if c in df.columns]

    sub = df[cols].copy()
    sub = sub.fillna(sub.mean(numeric_only=True)).fillna(0.0)

    n_pred, n_tp = len(cols), len(sub)
    header = {
        "FileVersion": 1,
        "NrOfPredictors": n_pred,
        "NrOfDataPoints": n_tp,
        "IncludesConstant": 0,
        "FirstConfoundPredictor": 1,
    }
    data_sdm = []
    for name in cols:
        data_sdm.append({
            "NameOfPredictor":    name,
            "ColorOfPredictor":   [120, 120, 120],
            "ValuesOfPredictor":  sub[name].to_numpy(dtype=np.float32).tolist(),
        })
    bvbabel.sdm.write_sdm(str(sdm_path), header, data_sdm)
    return n_pred

## 5. Per-subject driver

Auto-discovers fMRIPrep's `desc-confounds_timeseries.tsv` (handles ses-XX nesting) and the matching behavioural CSV (handles randomized category-to-run assignment).

In [6]:
def find_confounds_tsv(sub_id: str, run: int):
    """Locate the confounds TSV anywhere under FMRIPREP_DIR/sub-XXX/."""
    sub_root = FMRIPREP_DIR / sub_id
    if not sub_root.exists():
        return None
    glob = f"**/{sub_id}*_task-{TASK_LABEL}_run-{run:02d}_desc-confounds_timeseries.tsv"
    hits = sorted(sub_root.glob(glob))
    return hits[0] if hits else None


def process_subject(sub_num):
    """Process one subject's high-level task: write SDM + PRT for each run."""
    sub_id = f"sub-{sub_num}"
    sub_out = BV_DIR / sub_id
    sub_out.mkdir(parents=True, exist_ok=True)

    print(f"\n=== {sub_id} ===")
    for run in RUNS:
        rr = f"run-{run:02d}"
        conf_tsv = find_confounds_tsv(sub_id, run)
        behav_csv, category = find_behav_csv(BEHAV_DIR, sub_num, run)

        missing = []
        if conf_tsv is None:
            missing.append("confounds TSV")
        if behav_csv is None:
            missing.append("behav CSV")
        if missing:
            print(f"  [SKIP {rr}] missing: {', '.join(missing)}")
            continue

        if category not in CATEGORIES:
            print(f"  [SKIP {rr}] unrecognized category '{category}' in {behav_csv.name}")
            continue

        stem    = f"{sub_id}_task-{TASK_LABEL}_{rr}"
        sdm_out = sub_out / f"{stem}_confounds.sdm"
        prt_out = sub_out / f"{stem}.prt"

        try:
            n_conf  = confounds_to_sdm(conf_tsv, sdm_out)
            n_block = behav_to_prt(behav_csv, prt_out, category)
            print(f"  [{rr}]  category={category:5s}  "
                  f"PRT: {n_block} blocks across {len(SF_LEVELS)} SF conds  "
                  f"SDM: {n_conf} confounds")
        except Exception as e:
            print(f"  [ERROR {rr}] {type(e).__name__}: {e}")

## 6. Smoke test on sub-122 (the example you uploaded)

Verifies that:
- File patterns match what's actually on disk
- Categories are correctly extracted from filenames
- PRT condition labels are like `LSF-Body`, `HSF-Body`, `NSF-Body` etc.
- SDM has roughly the expected number of regressors

In [7]:
process_subject(122)


=== sub-122 ===
  [run-01]  category=Body   PRT: 15 blocks across 3 SF conds  SDM: 62 confounds
  [run-02]  category=House  PRT: 15 blocks across 3 SF conds  SDM: 48 confounds
  [run-03]  category=Face   PRT: 15 blocks across 3 SF conds  SDM: 58 confounds


## 7. Full batch

In [8]:
from datetime import datetime
t0 = datetime.now()
for sub_num in SUBJECTS:
    try:
        process_subject(sub_num)
    except Exception as e:
        print(f"  ERROR sub-{sub_num}: {type(e).__name__}: {e}")
print(f"\nDone in {datetime.now() - t0}")


=== sub-102 ===
  [run-01]  category=House  PRT: 15 blocks across 3 SF conds  SDM: 38 confounds
  [run-02]  category=Face   PRT: 15 blocks across 3 SF conds  SDM: 23 confounds
  [run-03]  category=Body   PRT: 15 blocks across 3 SF conds  SDM: 31 confounds

=== sub-103 ===
  [run-01]  category=Face   PRT: 15 blocks across 3 SF conds  SDM: 45 confounds
  [run-02]  category=House  PRT: 15 blocks across 3 SF conds  SDM: 56 confounds
  [run-03]  category=Body   PRT: 15 blocks across 3 SF conds  SDM: 90 confounds

=== sub-104 ===
  [run-01]  category=House  PRT: 15 blocks across 3 SF conds  SDM: 109 confounds
  [run-02]  category=Body   PRT: 15 blocks across 3 SF conds  SDM: 88 confounds
  [run-03]  category=Face   PRT: 15 blocks across 3 SF conds  SDM: 129 confounds

=== sub-105 ===
  [run-01]  category=House  PRT: 15 blocks across 3 SF conds  SDM: 34 confounds
  [run-02]  category=Body   PRT: 15 blocks across 3 SF conds  SDM: 21 confounds
  [run-03]  category=Face   PRT: 15 blocks across 

## 8. Sanity check — category counts and predictor coverage

Confirms each subject got 1 of each category across their 3 runs (otherwise the random assignment was unbalanced) and that all 9 unique condition labels appear across the cohort.

In [9]:
rows = []
for sub_num in SUBJECTS:
    sub_id = f"sub-{sub_num}"
    sub_dir = BV_DIR / sub_id
    row = {"sub": sub_num}
    for run in RUNS:
        _, cat = find_behav_csv(BEHAV_DIR, sub_num, run)
        row[f"run-{run:02d}"] = cat or "-"
    row["sdm_count"] = len(list(sub_dir.glob("*_confounds.sdm"))) if sub_dir.exists() else 0
    row["prt_count"] = len(list(sub_dir.glob("*.prt")))           if sub_dir.exists() else 0
    rows.append(row)
summary = pd.DataFrame(rows)
print("Run-to-category assignments per subject:\n")
print(summary.to_string(index=False))

# Verify each subject saw each category exactly once
summary["all_three_categories"] = summary[[f"run-{r:02d}" for r in RUNS]].apply(
    lambda x: sorted(x.tolist()) == sorted(CATEGORIES), axis=1
)
n_complete = summary["all_three_categories"].sum()
print(f"\nSubjects with all 3 categories present: {n_complete}/{len(summary)}")
if n_complete < len(summary):
    print("\nSubjects with incomplete category coverage:")
    print(summary[~summary["all_three_categories"]].to_string(index=False))

Run-to-category assignments per subject:

 sub run-01 run-02 run-03  sdm_count  prt_count
 102  House   Face   Body          6          6
 103   Face  House   Body          6          6
 104  House   Body   Face          6          6
 105  House   Body   Face          6          6
 106  House   Body   Face          6          6
 107   Face   Body  House          6          6
 108   Face  House   Body          6          6
 109  House   Face   Body          6          6
 110   Body  House   Face          6          6
 111   Body   Face  House          6          6
 112   Face   Body  House          6          6
 113   Body   Face  House          6          6
 115   Body   Face  House          6          6
 116   Body  House   Face          6          6
 117   Face   Body  House          6          6
 118   Face  House   Body          6          6
 119   Body  House   Face          6          6
 120   Face  House   Body          6          6
 121   Face  House   Body          6          

## 9. Inspect a generated PRT (optional)

Reads back one of the PRTs to confirm the condition labels are in the `<SF>-<Category>` form.

In [10]:
test_prt = BV_DIR / "sub-122" / f"sub-122_task-{TASK_LABEL}_run-01.prt"
if test_prt.exists():
    hdr, conds = bvbabel.prt.read_prt(str(test_prt))
    print(f"Condition labels in {test_prt.name}:")
    for c in conds:
        n_blocks = len(np.atleast_1d(c["Time start"]))
        print(f"  {c['NameOfCondition']:20s}  {n_blocks} blocks")
else:
    print(f"PRT not found: {test_prt}")

Condition labels in sub-122_task-SFhigh_run-01.prt:
  LSF-Body              5 blocks
  HSF-Body              5 blocks
  NSF-Body              5 blocks


## Next steps

1. **Import VTCs in BV** for the SFhigh task. Same workflow as SFlow: open the MNI VMR, then `File → Open` each NIfTI to create matching `.vtc` files. They should be saved as e.g. `sub-122_ses-01_task-SFhigh_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.vtc`.

2. **Build combined design matrices** (task + confounds) per run, using the same `build_combined_sdm` pipeline as before — just point it at the new `sub-XXX_task-SFhigh_run-RR.prt` and `..._confounds.sdm` files. The output goes to `..._design.sdm` (or `_design-taskonly.sdm` etc.). Each will have **3 task predictors + N confounds** (since each run has only 3 SF conditions for its single category).

3. **Run first-level GLMs** via the BV JS script. Each run's GLM will have 3 task betas (one per SF level for that run's category).

4. **Build MDM and run multi-study GLM**. With "Separate Predictors per Subject" on, BV pools across runs sharing predictor names. Since the predictor names are category-specific (`LSF-Body`, etc.), the Body run from sub-102 (run 1) gets pooled with the Body run from sub-103 (run 2). Each subject ends up with **9 task betas** (3 SF × 3 categories).

5. **ANCOVA Random Effects Analysis** with:
   - Within-subjects factor A: **Category** (3 levels: Body / Face / House)
   - Within-subjects factor B: **SF** (3 levels: LSF / NSF / HSF)
   - Between-subjects factor C: **Group** (BDD / HC)
   - 3×3×2 mixed factorial — yields main effects, two-way, and three-way interactions for testing your group hypotheses on the high-level task.

In [15]:
import bvbabel
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import gamma as gamma_fn


def double_gamma_hrf(t, a1=6.0, b1=1.0, a2=16.0, b2=1.0, c=1/6.0):
    """SPM-style canonical double-gamma HRF, sampled at times t (seconds)."""
    h = ((t**a1 * np.exp(-t/b1)) / (b1**(a1+1) * gamma_fn(a1+1))
         - c * (t**a2 * np.exp(-t/b2)) / (b2**(a2+1) * gamma_fn(a2+1)))
    h[t < 0] = 0
    return h


def get_n_timepoints(vtc_path: Path) -> int:
    """Read the number of TRs from a VTC header (faster than loading the data)."""
    header, _ = bvbabel.vtc.read_vtc(str(vtc_path))
    return int(header["Nr time points"])


def build_combined_sdm(prt_path: Path, sdm_path: Path, out_path: Path,
                       n_timepoints: int, tr_seconds: float) -> None:
    """Combine PRT (HRF-convolved task) + confound SDM into a single SDM.

    PRT condition names are passed through unchanged, so for the high-level task
    they'll be e.g. 'LSF-Body', 'HSF-Body', 'NSF-Body' and 3 task predictors will
    be written (one per SF for that run's category).
    """
    # --- Task predictors from PRT ------------------------------------------
    prt_header, prt_conds = bvbabel.prt.read_prt(str(prt_path))

    fine_dt  = 0.05
    duration = n_timepoints * tr_seconds
    fine_t   = np.arange(0, duration, fine_dt)
    hrf      = double_gamma_hrf(np.arange(0, 32, fine_dt))

    task_predictors = []
    for cond in prt_conds:
        stick  = np.zeros_like(fine_t)
        starts = np.atleast_1d(cond["Time start"]).astype(float) / 1000.0
        stops  = np.atleast_1d(cond["Time stop"]).astype(float)  / 1000.0
        for s, e in zip(starts, stops):
            stick[int(s/fine_dt):int(e/fine_dt)] = 1.0
        conv = np.convolve(stick, hrf)[:len(fine_t)]
        tr_idx = ((np.arange(n_timepoints) + 0.5) * tr_seconds / fine_dt).astype(int)
        tr_idx = np.clip(tr_idx, 0, len(conv) - 1)
        task_predictors.append({
            "NameOfPredictor":   cond["NameOfCondition"],
            "ColorOfPredictor":  list(cond["Color"]),
            "ValuesOfPredictor": conv[tr_idx].astype(np.float32).tolist(),
        })

    # --- Confound predictors from existing SDM -----------------------------
    sdm_header, sdm_predictors = bvbabel.sdm.read_sdm(str(sdm_path))

    # --- Combine and write -------------------------------------------------
    all_predictors = task_predictors + sdm_predictors
    n_pred = len(all_predictors)

    combined_header = {
        "FileVersion":            1,
        "NrOfPredictors":         n_pred,
        "NrOfDataPoints":         n_timepoints,
        "IncludesConstant":       0,
        "FirstConfoundPredictor": len(task_predictors) + 1,
    }
    bvbabel.sdm.write_sdm(str(out_path), combined_header, all_predictors)
    print(f"  [DESIGN]  {out_path.name}  {n_timepoints} TPs x {n_pred} predictors "
          f"({len(task_predictors)} task + {len(sdm_predictors)} confounds)")


# --- Build combined SDMs for every high-level subject/run -------------------
print(f"Building combined design matrices for task-{TASK_LABEL} (TR={TR_SECONDS}s)\n")

#for sub_num in SUBJECTS:
sub_num = 121
sub_dir = BV_DIR / f"sub-{sub_num}"
# if not sub_dir.exists():
#     continue
print(f"=== sub-{sub_num} ===")
for run in RUNS:
    rr  = f"run-{run:02d}"
    vtc = sub_dir / (f"sub-{sub_num}_ses-01_task-{TASK_LABEL}_{rr}"
                        f"_space-MNI152NLin2009cAsym_desc-preproc_bold.vtc")
    prt = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}.prt"
    sdm = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_confounds.sdm"
    out = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_design.sdm"

    missing = [p.name for p in (vtc, prt, sdm) if not p.exists()]
    if missing:
        print(f"  [SKIP {rr}] missing: {', '.join(missing)}")
        continue

    try:
        n_tp = get_n_timepoints(vtc)
        build_combined_sdm(prt, sdm, out, n_tp, TR_SECONDS)
    except Exception as e:
        print(f"  [ERROR {rr}] {type(e).__name__}: {e}")

Building combined design matrices for task-SFhigh (TR=1.0s)

=== sub-121 ===
  [SKIP run-01] missing: sub-121_ses-01_task-SFhigh_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.vtc
  [DESIGN]  sub-121_task-SFhigh_run-02_design.sdm  335 TPs x 64 predictors (3 task + 61 confounds)
  [DESIGN]  sub-121_task-SFhigh_run-03_design.sdm  335 TPs x 50 predictors (3 task + 47 confounds)


In [14]:
import re

# Path to the high-level behavioral CSV pattern, used for category lookup
BEHAV_PATTERN = re.compile(
    r"Subject_(?P<sub>\d+)_Run_(?P<run>\d+)_(?P<category>[A-Za-z]+)_RT\.csv$"
)

def get_run_category(sub_num, run):
    """Read the category for a specific subject's run from the CSV filename."""
    sub_dir = BEHAV_DIR / str(sub_num) / "high-level"
    if not sub_dir.exists():
        return None
    for f in sub_dir.iterdir():
        m = BEHAV_PATTERN.match(f.name)
        if m and int(m.group("sub")) == sub_num and int(m.group("run")) == run:
            return m.group("category")
    return None


def write_mdm(mdm_path, study_lines, label):
    """Write an MDM in BV's exact native format."""
    mdm_text = (
        "\n"
        "FileVersion:          3\n"
        "TypeOfFunctionalData: VTC\n\n"
        "RFX-GLM:              1\n\n"
        "PSCTransformation:    0\n"
        "zTransformation:      1\n"
        "SeparatePredictors:   2\n\n"
        f"NrOfStudies:          {len(study_lines)}\n"
        + "\n".join(study_lines) + "\n"
    )
    mdm_path.write_text(mdm_text)
    print(f"  Wrote {mdm_path.name}  ({len(study_lines)} studies)  [{label}]")


# --- Collect all (subject, run, category, vtc, sdm) tuples ------------------
records = []
for sub_num in SUBJECTS:
    sub_dir = BV_DIR / f"sub-{sub_num}"
    if not sub_dir.exists():
        continue
    for run in RUNS:
        rr  = f"run-{run:02d}"
        vtc = sub_dir / f"sub-{sub_num}_ses-01_task-{TASK_LABEL}_{rr}_space-MNI152NLin2009cAsym_desc-preproc_bold.vtc"
        sdm = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_design.sdm"
        if not (vtc.exists() and sdm.exists()):
            print(f"WARN: missing files for sub-{sub_num} {rr}")
            continue
        cat = get_run_category(sub_num, run)
        if cat is None:
            print(f"WARN: no category found for sub-{sub_num} {rr}")
            continue
        records.append((sub_num, run, cat, vtc, sdm))

print(f"\nCollected {len(records)} runs total\n")

# --- Build the four MDMs ------------------------------------------------------
print("Writing per-category MDMs (one run per subject per MDM):")
for category in CATEGORIES:
    cat_records = [r for r in records if r[2] == category]
    study_lines = [f'"{vtc}" "{sdm}"' for _, _, _, vtc, sdm in cat_records]
    out_path    = BV_DIR / f"group_RFX_{TASK_LABEL}_{category.lower()}.mdm"
    write_mdm(out_path, study_lines, f"category={category}")

print("\nWriting combined MDM (all runs, all categories):")
study_lines = [f'"{vtc}" "{sdm}"' for _, _, _, vtc, sdm in records]
out_path    = BV_DIR / f"group_RFX_{TASK_LABEL}_combined.mdm"
write_mdm(out_path, study_lines, "all categories combined")

WARN: missing files for sub-121 run-01

Collected 176 runs total

Writing per-category MDMs (one run per subject per MDM):
  Wrote group_RFX_SFhigh_body.mdm  (59 studies)  [category=Body]
  Wrote group_RFX_SFhigh_face.mdm  (58 studies)  [category=Face]
  Wrote group_RFX_SFhigh_house.mdm  (59 studies)  [category=House]

Writing combined MDM (all runs, all categories):
  Wrote group_RFX_SFhigh_combined.mdm  (176 studies)  [all categories combined]
